## Characterization of the experimental groups

In [12]:
import pandas as pd
import matplotlib.pyplot as plt
import datetime

# Constants
BARRELS_PER_TONNE = 7.33
TONNES_PER_MT = 1_000_000

In [33]:
MSCI_EM_constituent_list = [ "Brazil", "Chile", "China", "Colombia", "Czechia", "Egypt",
                             "Greece", "Hungary", "India", "Indonesia", "South Korea", "Kuwait", 
                             "Malaysia", "Mexico", "Peru", "Philippines", "Poland", "Qatar",
                            "Saudi Arabia", "South Africa", "Taiwan", "Thailand", "Turkey", "United Arab Emirates"] #Length: 24

### Valid CDS data

In [37]:
CDS_df = pd.read_csv('data/processed/CDS/Weekly_CDS.csv')
CDS_df['Date'] = pd.to_datetime(CDS_df['Date'])
CDS_df.set_index('Date', inplace=True)

CDS_df = CDS_df[CDS_df.index >= datetime.datetime(2014,1,1)].copy()
CDS_df = CDS_df[[col for col in CDS_df.columns if col in MSCI_EM_constituent_list]]

cds_stats = []

for country in CDS_df.columns:
    # Get the time series for this country (drop NaN values)
    series = CDS_df[country].dropna()
    
    if len(series) > 0:
        # Start date
        start_date = series.index[0]
        
        # Calculate returns (percentage change from one row to next)
        returns = series.diff()
        
        # Count zero returns (where consecutive values are the same)
        zero_returns = (returns == 0).sum()
        
        # Percentage of zero returns
        total_returns = len(returns) - 1  # Exclude the first NaN from diff()
        pct_zero_returns = (zero_returns / total_returns * 100) if total_returns > 0 else 0
        
        cds_stats.append({
            'Country': country,
            'Series Start': start_date,
            'Observations': len(series),
            'Zero Returns (%)': round(pct_zero_returns, 2)
        })

# Create and display the statistics table
cds_stats_df = pd.DataFrame(cds_stats).sort_values('Country')
print(cds_stats_df.to_string(index=False))
print(f"\nTotal countries: {len(cds_stats_df)}")

     Country Series Start  Observations  Zero Returns (%)
      Brazil   2014-01-03           574              0.35
       Chile   2014-01-03           574              0.52
       China   2014-01-03           574              1.57
    Colombia   2014-01-03           574              0.00
     Czechia   2014-01-03           574             16.58
       Egypt   2014-01-03           574              1.40
      Greece   2014-01-03           574             25.13
     Hungary   2014-01-03           574              4.36
       India   2015-03-27           510             41.65
   Indonesia   2014-01-03           574              0.87
      Kuwait   2017-06-16           394             32.32
    Malaysia   2014-01-03           574              1.57
      Mexico   2014-01-03           574              0.17
        Peru   2014-01-03           574              0.52
 Philippines   2014-01-03           574              1.92
      Poland   2014-01-03           574              7.33
       Qatar  

In [42]:
# Crude oil balance (need to handle apostrophe thousands separator)
crude_oil_balance = pd.read_csv('data/processed/Oil/crude_oil_trade_balance.csv', 
                                 thousands="'")
crude_oil_balance.set_index('Country', inplace=True)

crude_filtered = crude_oil_balance.loc[crude_oil_balance.index.isin(MSCI_EM_constituent_list)]

# Oil prices
annual_oil_price = pd.read_csv('data/processed/Oil/Annual_avg_price_of_oil_macrotrends.csv')
annual_oil_price['Year'] = pd.to_datetime(annual_oil_price['Date']).dt.year
oil_prices = annual_oil_price.set_index('Year')['Value']

# GDP (long format)
gdp_wide = pd.read_csv('data/processed/Macroeconomic_variables/annual_GDP_IMFWEO.csv')


crude_filtered = crude_filtered.T
crude_filtered.index = crude_filtered.index.astype(int)

# -----------------------------------------------------------------------------
# CONVERT MT TO USD BILLIONS
# -----------------------------------------------------------------------------

# After loading and filtering crude oil balance, convert to numeric
crude_filtered = crude_filtered.apply(pd.to_numeric, errors='coerce')

print(crude_filtered.dtypes)
print(crude_filtered.tail())
crude_oil_usd = crude_filtered.copy()
# Then do the conversion
for year in crude_oil_usd.index:
    if year in oil_prices.index:
        price = oil_prices[year]
        crude_oil_usd.loc[year] = crude_filtered.loc[year] * TONNES_PER_MT * BARRELS_PER_TONNE * price / 1e9
print("\nCrude oil in USD billions:")
print(crude_oil_usd.tail())

# -----------------------------------------------------------------------------
# PREPARE GDP (map country names and convert to billions)
# -----------------------------------------------------------------------------

# Filter to selected countries
gdp_filtered = gdp_wide.loc[gdp_wide.index.isin(MSCI_EM_constituent_list)]

# Transpose: index = years, columns = countries
gdp_transposed = gdp_filtered.T
gdp_transposed.index = gdp_transposed.index.astype(int)

# Convert to billions
gdp_transposed = gdp_transposed / 1e9

print("\nGDP in USD billions:")
print(gdp_transposed.tail())

# -----------------------------------------------------------------------------
# CALCULATE OIL BALANCE AS % OF GDP
# -----------------------------------------------------------------------------

common_years = crude_oil_usd.index.intersection(gdp_transposed.index)
common_countries = crude_oil_usd.columns.intersection(gdp_transposed.columns)

print(f"\nCommon years: {common_years.min()} - {common_years.max()}")
print(f"Common countries: {list(common_countries)}")

crude_aligned = crude_oil_usd.loc[common_years, common_countries]
gdp_aligned = gdp_transposed.loc[common_years, common_countries]

oil_balance_pct_gdp = (crude_aligned / gdp_aligned) * 100

print("\nOil Balance as % of GDP:")
print(oil_balance_pct_gdp.tail())

# -----------------------------------------------------------------------------
# PLOT
# -----------------------------------------------------------------------------

start_year = 2014
end_year = 2024

oil_balance_plot = oil_balance_pct_gdp.loc[start_year:end_year]

fig, ax = plt.subplots(figsize=(14, 8))

for country in MSCI_EM_constituent_list:
    if country in oil_balance_plot.columns:
        ax.plot(oil_balance_plot.index, oil_balance_plot[country], 
                's--', color='blue', linewidth=1.5, markersize=4, label=country)

ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Crude Oil Balance (% of GDP)', fontsize=12)
ax.set_title(f'Crude Oil Trade Balance as % of GDP ({start_year}-{end_year})\nRed = Oil Exporters, Blue = Controls', fontsize=14)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# -----------------------------------------------------------------------------
# SUMMARY TABLE
# -----------------------------------------------------------------------------

recent_years = [y for y in range(start_year, end_year + 1) if y in oil_balance_pct_gdp.index]
avg_balance = oil_balance_pct_gdp.loc[recent_years].mean().sort_values(ascending=True)  # ascending so exporters (negative) at top

print("\n" + "="*60)
print(f"AVERAGE CRUDE OIL TRADE BALANCE (% of GDP), {start_year}-{end_year}")
print("="*60)

for country in avg_balance.index:
    group = "EXPORTER" if country in oil_exporters else "CONTROL"
    print(f"{country:20s}: {avg_balance[country]:+.2f}%  [{group}]")


Country
Czechia                 float64
Poland                  float64
Turkey                  float64
Brazil                  float64
Chile                   float64
Colombia                float64
Mexico                  float64
China                   float64
India                   float64
Indonesia               float64
Malaysia                float64
Philippines             float64
South Korea             float64
Taiwan                  float64
Thailand                float64
Egypt                   float64
South Africa            float64
Kuwait                  float64
Qatar                   float64
Saudi Arabia            float64
United Arab Emirates    float64
dtype: object
Country  Czechia  Poland  Turkey  Brazil  Chile  Colombia  Mexico  China  \
2020         6.2    25.0    28.8   -61.6    7.5     -28.6   -59.0  540.4   
2021         6.8    24.2    31.0   -56.3    8.2     -24.3   -53.4  510.3   
2022         7.4    27.3    33.2   -55.7    7.4     -20.2   -49.2  506.2   
20

ValueError: invalid literal for int() with base 10: 'COUNTRY'

In [31]:
common_countries

Index(['Poland', 'Brazil', 'Chile', 'Colombia', 'Mexico', 'Indonesia',
       'Philippines', 'Egypt', 'Qatar'],
      dtype='str')